In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import os
import pandas as pd 
from time import sleep
from bs4 import BeautifulSoup
from selenium import webdriver
import datetime
from urllib.parse import urljoin
import requests
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

#from webdriver_manager.chrome import ChromeDriverManager


In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'SE FI' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.2")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

# scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

# %%


Running SE FI Web Scraping Tool v.1.2


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()


In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={
        # 'SE FI 1':{"huvudkategori": "Bank"}, 

        # 'SE FI 5': {"huvudkategori": "Bank, utländska"}, 

        # 'SE FI 7': {"huvudkategori": "Betaltjänstföretag"} , 

        # 'SE FI 10': {"huvudkategori": "Betaltjänstföretag,+utländska"}, 

        # 'SE FI 13': {"huvudkategori": 'Fondbolag/AIF-förvaltare'}, 

        # 'SE FI 16': {"huvudkategori": 'Fond'}, 

        # 'SE FI 18': {"huvudkategori": 'Fondföretag/bolag/AIF-förvaltare,+utländska'}, 

        # # #  #'SE FI 20': 'https://www.fi.se/en/our-registers/company-register/?huvudkategori=Fond%2C+utl%C3%A4ndska&area=#results', 
         
        # 'SE FI 1 20':  {"huvudkategori": "Fond, utländska", "cat": "UTAIF"}, 
         
        # # #  'SE FI 2 20': 'https://www.fi.se/en/our-registers/company-register/?huvudkategori=Fond%2C+utl%C3%A4ndska&cat=UTAIFU&area=#results',  # Foreign Non-Authorised Remove info
         
        # 'SE FI 3 20':  {"huvudkategori": "Fond, utländska", "cat": "UTDELF"}, 
         
        'SE FI 4 20': {"huvudkategori": "Fond, utländska", "cat": "UTF1:7"},  
         
        # 'SE FI 21':  {"huvudkategori": "Försäkringsföretag"}, 

        # 'SE FI 24': {"huvudkategori": "Försäkringsföretag, utländska"}, 

        # 'SE FI 25': {"huvudkategori":'Konsumentkreditinstitut'}, 

        # 'SE FI 27': {"huvudkategori":'Hypoteksinstitut'}, 
        # 'SE FI 28': {"huvudkategori":'Kreditmarknadsföretag'}, 
        # 'SE FI 30': {"huvudkategori":'Kreditmarknadsföretag, utländska'}, 
        # 'SE FI 32': {"huvudkategori":'Utgivare av elektroniska pengar'},
        # 'SE FI 35': {"huvudkategori":'Värdepappersbolag'},
        # 'SE FI 36': {"huvudkategori":'Värdepappersbolag, utländska'},
        # 'SE FI 39': {"huvudkategori":'Övriga institut'},
 
        }



Typology={

       regulatorName + ' 1': 'Banks',
       regulatorName + ' 5': 'Banks, foreign',
       regulatorName + ' 7': 'Payment services companies',
       regulatorName + ' 10': '	Payment services companies, foreign',
       regulatorName + ' 13': 'Fund companies/AIF Managers',
       regulatorName + ' 16': '	Funds',
       regulatorName + ' 18': 'Fund companies/AIF Managers, foreign',
       regulatorName + ' 1'+' 20': 'Funds, foreign',
       regulatorName + ' 2'+' 20': 'Funds, foreign', # Foreign Non-Authorised
       regulatorName + ' 3'+' 20': 'Funds, foreign',
       regulatorName + ' 4'+' 20': 'Funds, foreign',
       regulatorName + ' 21': 'Insurance companies, nationwide companies',
       regulatorName + ' 24': 'Insurance companies, foreign',
       regulatorName + ' 25': 'Consumer Credit Companies',
       regulatorName + ' 27': 'Mortgage institutes',
       regulatorName + ' 28': 'Credit-market companies',
       regulatorName + ' 30': 'Credit-market companies, foreign',
       regulatorName + ' 32': 'Electronic money issuers',
       regulatorName + ' 35': 'Investment firms',
       regulatorName + ' 36': 'Investment firms, foreign',
       regulatorName + ' 39': 'Other companies',

        }




sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

          'Phone - Mother company': [], 'Check': []}



ISO= {"": "", 'NAN': '', 'OTHER': '', "AFGHANISTAN": "AF", "ÅLAND ISLANDS": "AX", "ALBANIA": "AL", "ALGERIA": "DZ", "AMERICAN SAMOA": "AS", "ANDORRA": "AD", "ANGOLA": "AO", "ANGUILLA": "AI", "ANTARCTICA": "AQ", "ANTIGUA AND BARBUDA": "AG", "ARGENTINA": "AR", "ARMENIA": "AM", "ARUBA": "AW", "AUSTRALIA": "AU", "AUSTRIA": "AT", "AZERBAIJAN": "AZ", "BAHAMAS, THE": "BS", "BAHRAIN": "BH", "BANGLADESH": "BD", "BARBADOS": "BB", "BELARUS": "BY", "BELGIUM": "BE", "BELIZE": "BZ", "BENIN": "BJ", "BERMUDA": "BM", "BHUTAN": "BT", "BOLIVIA": "BO", "BONAIRE, SINT EUSTATIUS AND SABA": "BQ", "BOSNIA AND HERZEGOVINA": "BA", "BOTSWANA": "BW", "BOUVET ISLAND": "BV", "BRAZIL": "BR", "BRITISH INDIAN OCEAN TERRITORY": "IO", "BRUNEI": "BN", "BULGARIA": "BG", "BURKINA FASO": "BF", "BURUNDI": "BI", "CABO VERDE": "CV", "CAMBODIA": "KH", "CAMEROON, UNITED REPUBLIC OF": "CM", "CANADA": "CA", "CAYMAN ISLANDS": "KY", "CENTRAL AFRICAN REPUBLIC": "CF", "CHAD": "TD", "CHILE": "CL", "CHINA, PEOPLES REPUBLIC OF": "CN","CHINA": "CN", "CHRISTMAS ISLAND": "CX", "COCOS (KEELING) ISLANDS": "CC", "COLOMBIA": "CO", "COMOROS": "KM", "CONGO": "CG", "CONGO, DEMOCRATIC REPUBLIC OF THE": "CD", "COOK ISLANDS": "CK", "COSTA RICA": "CR", "CÔTE D'IVOIRE": "CI", "CROATIA": "HR", "CUBA": "CU", "CURACAO, BONAIRE, SABA, ST. MARTIN & ST.": "CW", "CYPRUS": "CY", "CZECH REPUBLIC": "CZ", "DENMARK": "DK", "DJIBOUTI": "DJ", "DOMINICA": "DM", "DOMINICAN REPUBLIC": "DO", "ECUADOR": "EC", "EGYPT": "EG", "EL SALVADOR": "SV", "EQUATORIAL GUINEA": "GQ", "ERITREA": "ER", "ESTONIA": "EE", "ESWATINI": "SZ", "ETHIOPIA": "ET", "FALKLAND ISLANDS (MALVINAS)": "FK", "FAROE ISLANDS": "FO", "FIJI": "FJ", "FINLAND": "FI", "FRANCE": "FR", "FRENCH GUIANA": "GF", "FRENCH POLYNESIA": "PF", "FRENCH SOUTHERN TERRITORIES": "TF", "GABON": "GA", "GAMBIA": "GM", "GEORGIA": "GE", 'GEORGIA/GRUZINSKAYA': 'GE', "GERMANY": "DE", "GHANA": "GH", "GIBRALTAR": "GI", "GREECE": "GR", "GREENLAND": "GL", "GRENADA": "GD", "GUADELOUPE": "GP", "GUAM": "GU", "GUATEMALA": "GT", "GUERNSEY": "GG", "GUINEA": "GN", "GUINEA-BISSAU": "GW", "GUYANA": "GY", "HAITI": "HT", "HEARD ISLAND AND MCDONALD ISLANDS": "HM", "HOLY SEE": "VA", "HONDURAS": "HN", "HONG KONG": "HK", "HUNGARY": "HU", "ICELAND": "IS", "INDIA": "IN", "INDONESIA": "ID", "IRAN": "IR", "IRAQ": "IQ", "IRELAND": "IE", "ISLE OF MAN": "IM", "ISRAEL": "IL", "ITALY": "IT", "JAMAICA": "JM", "JAPAN": "JP", "JERSEY": "JE", "JORDAN": "JO", "KAZAKHSTAN": "KZ", "KENYA": "KE", "KIRIBATI": "KI", """KOREA (DEMOCRATIC PEOPLE"S REPUBLIC OF)""": "KP", "KOREA, SOUTH": "KR", "KUWAIT": "KW", "KYRGYZSTAN": "KG", "LAO PEOPLE'S DEMOCRATIC REPUBLIC": "LA", "LATVIA": "LV", "LEBANON": "LB", "LESOTHO": "LS", "LIBERIA": "LR", "LIBYA": "LY", "LIECHTENSTEIN": "LI", "LITHUANIA": "LT", "LUXEMBOURG": "LU", "MACAU": "MO", "MADAGASCAR": "MG", "MALAWI": "MW", "MALAYSIA": "MY", "MALDIVES": "MV", "MALI": "ML", "MALTA": "MT", "MARSHALL ISLANDS": "MH", "MARTINIQUE": "MQ", "MAURITANIA": "MR", "MAURITIUS": "MU", "MAYOTTE": "YT", "MEXICO": "MX", "FEDERATED STATES OF MICRONESIA": "FM", "MOLDOVA, REPUBLIC OF": "MD", "MONACO": "MC", "MONGOLIA": "MN", "MONTENEGRO": "ME", "MONTSERRAT": "MS", "MOROCCO": "MA", "MOZAMBIQUE": "MZ", "MYANMAR": "MM", "NAMIBIA": "NA", "NAURU": "NR", "NEPAL": "NP", "NETHERLANDS": "NL", "NEW CALEDONIA": "NC", "NEW ZEALAND": "NZ", "NICARAGUA": "NI", "NIGER": "NE", "NIGERIA": "NG", "NIUE": "NU", "NORFOLK ISLAND": "NF", "NORTH MACEDONIA": "MK", "NORTHERN MARIANA ISLANDS": "MP", "NORWAY": "NO", "OMAN": "OM", "PAKISTAN": "PK", "PALAU": "PW", "PALESTINE, STATE OF": "PS", "PANAMA": "PA", "PAPUA NEW GUINEA": "PG", "PARAGUAY": "PY", "PERU": "PE", "PHILIPPINES": "PH", "PITCAIRN": "PN", "POLAND": "PL", "PORTUGAL": "PT", "PUERTO RICO": "PR", "QATAR": "QA", "RÉUNION": "RE", "ROMANIA": "RO", "RUSSIA": "RU", "RWANDA": "RW", "SAINT BARTHÉLEMY": "BL", "SAINT HELENA, ASCENSION AND TRISTAN DA CUNHA": "SH", "SAINT KITTS AND NEVIS": "KN", "SAINT LUCIA": "LC", "SAINT MARTIN (FRENCH PART)": "MF", "SAINT PIERRE AND MIQUELON": "PM", "SAINT VINCENT AND THE GRENADINES": "VC", "SAMOA": "WS", "SAN MARINO": "SM", "SAO TOME AND PRINCIPE": "ST", "SAUDI ARABIA": "SA", "SENEGAL": "SN", "SERBIA": "RS", "SEYCHELLES": "SC", "SIERRA LEONE": "SL", "SINGAPORE": "SG", "SINT MAARTEN (DUTCH PART)": "SX", "SLOVAKIA": "SK", "SLOVAK REPUBLIC": "SK", "SLOVENIA": "SI", "SOLOMON ISLANDS": "SB", "SOMALIA": "SO", "SOUTH AFRICA": "ZA", "SOUTH GEORGIA AND THE SOUTH SANDWICH ISLANDS": "GS", "SOUTH SUDAN": "SS", "SPAIN": "ES", "SRI LANKA": "LK", "SUDAN": "SD", "SURINAME": "SR", "SVALBARD AND JAN MAYEN": "SJ", "SWEDEN": "SE", "SWITZERLAND": "CH", "SYRIAN ARAB REPUBLIC": "SY", "TAIWAN": "TW",'TAIWAN,  REPUBLIC OF CHINA': 'TW' ,"TAJIKISTAN": "TJ", "TANZANIA, UNITED REPUBLIC OF": "TZ", "THAILAND": "TH", "TIMOR-LESTE": "TL", "TOGO": "TG", "TOKELAU": "TK", "TONGA": "TO", "TRINIDAD AND TOBAGO": "TT", "TUNISIA": "TN", "TURKEY": "TR", "TURKMENISTAN": "TM", "TURKS & CAICOS ISLANDS": "TC", "TUVALU": "TV", "UGANDA": "UG", "UKRAINE": "UA", "UNITED ARAB EMIRATES": "AE", "UNITED KINGDOM OF GREAT BRITAIN AND NORTHERN IRELAND": "GB", "UNITED STATES": "US", "UNITED STATES MINOR OUTLYING ISLANDS": "UM", "URUGUAY": "UY", "UZBEKISTAN": "UZ", "VANUATU": "VU", "VENEZUELA": "VE", "VIETNAM": "VN", "BRITISH VIRGIN ISLANDS": "VG", "VIRGIN ISLANDS OF THE U.S.": "VI", "WALLIS AND FUTUNA": "WF", "WESTERN SAHARA": "EH", "YEMEN": "YE", "ZAMBIA": "ZM", "ZIMBABWE": "ZW", "ENGLAND": "GB", "UNITED KINGDOM": "GB", "UNITED KINGDOM (OTHER)": "GB", "FRANCE (OTHER)": "FR", "WALES": "GB", "CONGO (KINSHASA)": "CD", "CONGO (BRAZZAVILLE)": "CD", "SCOTLAND": "GB", "ITALY (OTHER)": "IT", "INDONESIA (OTHER)": "ID", "INDIA (OTHER)": "IN", "MOROCCO (OTHER)": "MA", "NEW ZEALAND (OTHER)": "NZ", "SWITZERLAND (OTHER)": "CH", "MALAYSIA (OTHER)": "MY", "NETHERLANDS ANTILLES": "AN", "TRINIDAD & TOBAGO (OTHER)": "TT", "CHANNEL ISLANDS": "GB", "UNITED ARAB EMIRATES (OTHER)": "AE", "DENMARK (OTHER)": "DK", "COMORO ISLANDS": "KM", "MACEDONIA (FORMER YUGOSLAV REPUBLIC OF)": "MK", "SERBIA AND MONTENEGRO(FORMER YUGOSLAVIA)": "CS", "TRINIDAD": "TT", "ETHIOPIA (OTHER)": "ET", "IVORY COAST": "CI", "DUBAI": "AE", "BRITISH WEST INDIES (OTHER)": "VG", "SWAZILAND": "SZ", 'UNITED KINGDOM  (OTHER)': 'GB'}




processdate = now.strftime('%Y-%m-%d')



# %%


In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict

In [ ]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):

    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")

    url = "https://www.fi.se/en/our-registers/company-register/"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36"
    }

    resp = requests.get(url, params=regdict[reg], headers=headers, timeout=30)
    resp.raise_for_status()
    print(resp.url)
    print(resp.status_code)
    soup = BeautifulSoup(resp.text, "html.parser")
    # example: find the results table
    table = soup.find("table", {"id": "institut"})
        
    tbody=table.find("tbody")

    trs=tbody.find_all("tr")

    for tr in trs :
        data = {}
        current = None
        if len(tr.find_all('td')) == 0:
            continue
        name_ = tr.find_all('td')[0].text.strip()
        #print(name_)
        href = tr.find_all("td")[0].find("a")["href"]
        cin = tr.find_all('td')[1].text.strip().lstrip()
        detail_url = urljoin(url, href)


        retry = Retry(
            total=5,
            connect=5,
            read=5,
            status=5,
            backoff_factor=1,  # 1s, 2s, 4s, 8s...
            status_forcelist=[404,429, 500, 502, 503, 504],
            allowed_methods=["GET"],
            raise_on_status=False
        )
        session = requests.Session()
        session.headers.update(headers)
        session.mount("https://", HTTPAdapter(max_retries=retry))
        session.mount("http://", HTTPAdapter(max_retries=retry))

        resp_detail = session.get(detail_url, headers=headers, timeout=30)
        soup_detail = BeautifulSoup(resp_detail.text, "html.parser")
        info_section  = soup_detail.find("dl", {"class":"funky"})


        for node in info_section.find_all(["dt", "dd"], recursive=False):
            if node.name == "dt":
                current = node.get_text(strip=True)
                data[current] = []
            elif node.name == "dd" and current:
                text = node.get_text(" ", strip=True)
                if text and text != "\xa0":
                    data[current].append(text)

        # Optionally collapse lists to strings:
        data = {k: " ".join(v) if len(v) > 1 else (v[0] if v else "") for k, v in data.items()}
        #print(data)
        address_ = data['Address']
        tele_ = data['Telephone']
        cate_ = data['Category']
        lei_ = data['LEI code']
        fii = data['FI identification number']
        # data['Status'].split(',')[-1].lstrip()
        # print(data)
        sqldict['Name'].append(name_)
        if cin:
            sqldict['InternalID_1'].append(cin)
            sqldict['InternalID_1_type'].append('Corporate identification number')
        sqldict['ListProcessDate'].append(processdate)
        sqldict['RegCtry'].append(reg.split()[0])
        sqldict['RegCode'].append(reg.split()[1])
        sqldict['ListCode'].append(reg.split()[-1])
        sqldict['ListName'].append(Typology[reg])
        sqldict['RegulationType'].append('Regulated')
        sqldict['Address_1'].append(address_)
        sqldict['Phone'].append(tele_ if tele_!='0000000'or tele_!='0000' or tele_!='00000' else '')
        sqldict['Typology'].append(cate_)
        sqldict['LEI Code'].append(lei_)
        sqldict['InternalID_2'].append(fii)
        sqldict['InternalID_2_type'].append('FI identification number')

        if address_:
            try:
                sqldict['City'].append(address_.split(' ')[-2] if len(address_.split(' ')[-2])>2 else '')
            except:
                sqldict['City'].append('')
            try:
                sqldict['Cntry'].append(address_.split(' ')[-1] if len(address_.split(' ')[-1])>1 else '')
            except:
                sqldict['Cntry'].append('')
        sqldict = bourange_same_length_array(sqldict)
        


[INFO] : Working 1/1 _(SE FI 4 20)_ 
https://www.fi.se/en/our-registers/company-register/?huvudkategori=Fond%2C+utl%C3%A4ndska&cat=UTF1%3A7
200


In [ ]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df = pd.DataFrame(sqldict)

s_city = df["City"].astype(str).str.strip()
s_country = df["Cntry"].astype(str).str.strip()

# City: bad if starts/ends with digit, starts with '.', ends with ',' or ends with '-'
bad_city = s_city.str.match(r"^\d|.*\d$|^\.|.*,$|.*-$")
s_city = s_city.mask(bad_city, "")

# Country: bad if starts/ends with digit, starts with '.', ends with ','
bad_country = s_country.str.match(r"^\d|.*\d$|^\.|.*,$")
s_country = s_country.mask(bad_country, "")

# If wrapped in parentheses, strip them
s_city = s_city.str.replace(r"^\((.*)\)$", r"\1", regex=True)

df["City"] = s_city
df["Cntry"] = s_country


df.to_excel(filename, index=False)

driver.quit()

sleep(3)
